In [4]:
import psycopg2
import tkinter as tk
from tkinter import messagebox, ttk

# Connect to PostgreSQL
connection = psycopg2.connect(
    host="localhost",
    database="bookstore",
    user="postgres",
    password="Bakhrom2004" 
)
cursor = connection.cursor()

# Exit Application
def exit_app():
    connection.close()
    root.destroy()

# Add Book (with update if exists)
def add_book_window():
    window = tk.Toplevel(root)
    window.title("Add Book")
    window.geometry("400x400")

    tk.Label(window, text="Title").pack()
    title_entry = tk.Entry(window)
    title_entry.pack()

    tk.Label(window, text="Author").pack()
    author_entry = tk.Entry(window)
    author_entry.pack()

    tk.Label(window, text="Price").pack()
    price_entry = tk.Entry(window)
    price_entry.pack()

    tk.Label(window, text="Quantity").pack()
    quantity_entry = tk.Entry(window)
    quantity_entry.pack()

    tk.Label(window, text="Category").pack()
    cursor.execute("SELECT category_id, category_name FROM Categories ORDER BY category_name")
    categories = cursor.fetchall()
    category_var = tk.StringVar()
    category_dropdown = ttk.Combobox(window, textvariable=category_var)
    category_dropdown['values'] = [f"{cat[0]} - {cat[1]}" for cat in categories]
    category_dropdown.pack()

    def save_book():
        title = title_entry.get()
        author = author_entry.get()
        price = float(price_entry.get())
        quantity = int(quantity_entry.get())
        category_id = int(category_var.get().split(' - ')[0])

        cursor.execute("SELECT book_id FROM books WHERE title = %s AND author = %s", (title, author))
        existing = cursor.fetchone()

        if existing:
            choice = messagebox.askyesno("Book Exists", "This book exists. Update quantity and price?")
            if choice:
                cursor.execute(
                    "UPDATE books SET quantity_in_stock = quantity_in_stock + %s, price = %s WHERE book_id = %s",
                    (quantity, price, existing[0])
                )
                connection.commit()
                messagebox.showinfo("Updated", "Book updated successfully!")
            else:
                cursor.execute(
                    "INSERT INTO books (title, author, price, quantity_in_stock, category_id) VALUES (%s, %s, %s, %s, %s)",
                    (title, author, price, quantity, category_id)
                )
                connection.commit()
                messagebox.showinfo("Added", "New book added successfully!")
        else:
            cursor.execute(
                "INSERT INTO books (title, author, price, quantity_in_stock, category_id) VALUES (%s, %s, %s, %s, %s)",
                (title, author, price, quantity, category_id)
            )
            connection.commit()
            messagebox.showinfo("Added", "Book added successfully!")

        window.destroy()

    tk.Button(window, text="Save Book", command=save_book).pack(pady=20)

# Add Customer
def add_customer_window():
    window = tk.Toplevel(root)
    window.title("Add Customer")
    window.geometry("400x300")

    tk.Label(window, text="Name").pack()
    name_entry = tk.Entry(window)
    name_entry.pack()

    tk.Label(window, text="Phone Number").pack()
    phone_entry = tk.Entry(window)
    phone_entry.pack()

    tk.Label(window, text="Email").pack()
    email_entry = tk.Entry(window)
    email_entry.pack()

    def save_customer():
        cursor.execute(
            "INSERT INTO Customers (name, phone_number, email) VALUES (%s, %s, %s)",
            (name_entry.get(), phone_entry.get(), email_entry.get())
        )
        connection.commit()
        messagebox.showinfo("Success", "Customer added successfully!")
        window.destroy()

    tk.Button(window, text="Save Customer", command=save_customer).pack(pady=20)

# Add Employee
def add_employee_window():
    window = tk.Toplevel(root)
    window.title("Add Employee")
    window.geometry("400x300")

    tk.Label(window, text="Name").pack()
    name_entry = tk.Entry(window)
    name_entry.pack()

    tk.Label(window, text="Position").pack()
    position_entry = tk.Entry(window)
    position_entry.pack()

    tk.Label(window, text="Email").pack()
    email_entry = tk.Entry(window)
    email_entry.pack()

    def save_employee():
        cursor.execute(
            "INSERT INTO Employees (name, position, email) VALUES (%s, %s, %s)",
            (name_entry.get(), position_entry.get(), email_entry.get())
        )
        connection.commit()
        messagebox.showinfo("Success", "Employee added successfully!")
        window.destroy()

    tk.Button(window, text="Save Employee", command=save_employee).pack(pady=20)

# Place Order
def place_order_window():
    order_window = tk.Toplevel(root)
    order_window.title("Place Order")
    order_window.geometry("500x600")

    tk.Label(order_window, text="Select Customer").pack()
    cursor.execute("SELECT customer_id, name FROM Customers ORDER BY name")
    customers = cursor.fetchall()
    customer_var = tk.StringVar()
    customer_dropdown = ttk.Combobox(order_window, textvariable=customer_var)
    customer_dropdown['values'] = [f"{cust[0]} - {cust[1]}" for cust in customers]
    customer_dropdown.pack()

    tk.Label(order_window, text="Select Employee").pack()
    cursor.execute("SELECT employee_id, name FROM Employees ORDER BY name")
    employees = cursor.fetchall()
    employee_var = tk.StringVar()
    employee_dropdown = ttk.Combobox(order_window, textvariable=employee_var)
    employee_dropdown['values'] = [f"{emp[0]} - {emp[1]}" for emp in employees]
    employee_dropdown.pack()

    tk.Label(order_window, text="Order Date (YYYY-MM-DD)").pack()
    order_date_entry = tk.Entry(order_window)
    order_date_entry.pack()

    def save_order():
        customer_id = int(customer_var.get().split(' - ')[0])
        employee_id = int(employee_var.get().split(' - ')[0])
        order_date = order_date_entry.get()

        cursor.execute(
            "INSERT INTO Orders (customer_id, employee_id, order_date) VALUES (%s, %s, %s) RETURNING order_id",
            (customer_id, employee_id, order_date)
        )
        order_id = cursor.fetchone()[0]

        cursor.execute("SELECT book_id, title, quantity_in_stock, price FROM Books ORDER BY title")
        books = cursor.fetchall()
        book_window = tk.Toplevel(order_window)
        book_window.title("Select Books to Order")

        book_vars = {}
        qty_entries = {}

        for book in books:
            frame = tk.Frame(book_window)
            frame.pack(anchor='w')
            var = tk.IntVar()
            tk.Checkbutton(frame, text=f"{book[0]} - {book[1]} (Stock: {book[2]})", variable=var).pack(side='left')
            qty_entry = tk.Entry(frame, width=5)
            qty_entry.pack(side='left', padx=10)
            book_vars[book[0]] = (var, qty_entry, book[2], book[3])

        def save_order_items():
            for book_id, (var, qty_entry, stock, price) in book_vars.items():
                if var.get():
                    try:
                        quantity = int(qty_entry.get())
                        if quantity <= 0 or quantity > stock:
                            messagebox.showerror("Error", f"Invalid quantity for book ID {book_id}. Available: {stock}")
                            return
                        total_price = price * quantity
                        cursor.execute(
                            "INSERT INTO Order_Items (order_id, book_id, quantity, total_price) VALUES (%s, %s, %s, %s)",
                            (order_id, book_id, quantity, total_price)
                        )
                        cursor.execute(
                            "UPDATE Books SET quantity_in_stock = quantity_in_stock - %s WHERE book_id = %s",
                            (quantity, book_id)
                        )
                    except ValueError:
                        messagebox.showerror("Error", f"Invalid quantity for book ID {book_id}.")
                        return

            connection.commit()
            messagebox.showinfo("Success", "Order placed successfully!")
            book_window.destroy()
            order_window.destroy()

        tk.Button(book_window, text="Save Order", command=save_order_items).pack(pady=20)

    tk.Button(order_window, text="Next: Select Books", command=save_order).pack(pady=20)

# View Table (Generic with Search)
def view_table(table_name, search_column, columns):
    window = tk.Toplevel(root)
    window.title(f"View {table_name}")
    window.geometry("800x600")

    search_frame = tk.Frame(window)
    search_frame.pack(pady=10)
    search_entry = tk.Entry(search_frame)
    search_entry.pack(side='left', padx=5)
    search_button = tk.Button(search_frame, text="Search", command=lambda: refresh())
    search_button.pack(side='left')

    tree = ttk.Treeview(window, columns=columns, show='headings')
    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=100)
    tree.pack(expand=True, fill='both')

    def refresh():
        tree.delete(*tree.get_children())
        keyword = search_entry.get()
        if keyword:
            cursor.execute(f"SELECT * FROM {table_name} WHERE {search_column} ILIKE %s", ('%' + keyword + '%',))
        else:
            cursor.execute(f"SELECT * FROM {table_name}")
        for row in cursor.fetchall():
            tree.insert('', 'end', values=row)

    refresh()

# View Orders
def view_orders_window():
    window = tk.Toplevel(root)
    window.title("View Orders")
    window.geometry("1000x600")

    search_frame = tk.Frame(window)
    search_frame.pack(pady=10)
    search_entry = tk.Entry(search_frame)
    search_entry.pack(side='left', padx=5)
    search_button = tk.Button(search_frame, text="Search", command=lambda: refresh())
    search_button.pack(side='left')

    # Updated Columns
    tree = ttk.Treeview(window, columns=("OrderID", "Customer", "Employee", "Date", "Book", "Quantity", "UnitPrice", "TotalPrice"), show='headings')
    for col in ("OrderID", "Customer", "Employee", "Date", "Book", "Quantity", "UnitPrice", "TotalPrice"):
        tree.heading(col, text=col)
        tree.column(col, width=100)
    tree.pack(expand=True, fill='both')

    def refresh():
        tree.delete(*tree.get_children())
        keyword = search_entry.get()
        query = """
            SELECT o.order_id, c.name, e.name, o.order_date, b.title, oi.quantity,
                   ROUND(oi.total_price / oi.quantity, 2) AS unit_price,
                   oi.total_price
            FROM orders o
            JOIN customers c ON o.customer_id = c.customer_id
            JOIN employees e ON o.employee_id = e.employee_id
            JOIN order_items oi ON o.order_id = oi.order_id
            JOIN books b ON oi.book_id = b.book_id
        """
        if keyword:
            query += " WHERE c.name ILIKE %s"
            cursor.execute(query, ('%' + keyword + '%',))
        else:
            cursor.execute(query)

        for row in cursor.fetchall():
            tree.insert('', 'end', values=row)

    refresh()

# Main Window
root = tk.Tk()
root.title("Bookstore Management System")
root.geometry("700x800")

tk.Label(root, text="Bookstore Management System", font=("Arial", 18)).pack(pady=20)

tk.Button(root, text="Add Book", width=30, command=add_book_window).pack(pady=10)
tk.Button(root, text="Add Customer", width=30, command=add_customer_window).pack(pady=10)
tk.Button(root, text="Add Employee", width=30, command=add_employee_window).pack(pady=10)
tk.Button(root, text="Place Order", width=30, command=place_order_window).pack(pady=10)
tk.Button(root, text="View Books", width=30, command=lambda: view_table("Books", "title", ("BookID", "Title", "Author", "Price", "Stock", "CategoryID"))).pack(pady=10)
tk.Button(root, text="View Customers", width=30, command=lambda: view_table("Customers", "name", ("CustomerID", "Name", "Phone", "Email"))).pack(pady=10)
tk.Button(root, text="View Employees", width=30, command=lambda: view_table("Employees", "name", ("EmployeeID", "Name", "Position", "Email"))).pack(pady=10)
tk.Button(root, text="View Orders", width=30, command=view_orders_window).pack(pady=10)
tk.Button(root, text="Exit", width=30, command=exit_app).pack(pady=10)

root.mainloop()
